# PDF RAG with Groq and Free Local Embeddings

This notebook uses the local `BAAI/bge-small-en-v1.5` embedding model and Groq for answer generation.

In [ ]:
import json
import os
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

current_directory = Path.cwd()
PROJECT_DIRECTORY = (
    current_directory / 'rag-for-beginners'
    if (current_directory / 'rag-for-beginners' / 'docs').is_dir()
    else current_directory
)
PDF_PATH = PROJECT_DIRECTORY / 'docs' / '66---16-06-2021-02-48-53.pdf'
PERSIST_DIRECTORY = PROJECT_DIRECTORY / 'db' / 'chroma_pdf_bge_groq'
EMBEDDING_MODEL = 'BAAI/bge-small-en-v1.5'
GROQ_MODEL = 'openai/gpt-oss-120b'

## 1. Load and split the PDF

In [ ]:
if not PDF_PATH.is_file():
    raise FileNotFoundError(f'PDF file not found: {PDF_PATH}')

documents = PyPDFLoader(str(PDF_PATH)).load()
if not documents:
    raise FileNotFoundError(f'No pages found in {PDF_PATH}')

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)
print(f'Loaded {len(documents)} pages and created {len(chunks)} chunks.')

## 2. Create or load the local vector store

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

if PERSIST_DIRECTORY.is_dir():
    vector_store = Chroma(
        persist_directory=str(PERSIST_DIRECTORY),
        embedding_function=embeddings,
        collection_metadata={'hnsw:space': 'cosine'},
    )
else:
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=str(PERSIST_DIRECTORY),
        collection_metadata={'hnsw:space': 'cosine'},
    )

print(f'Vector store ready at {PERSIST_DIRECTORY}')

## 3. Query Groq with retrieved PDF context

In [ ]:
def ask_groq(question, context):
    groq_api_key = os.getenv('GROQ_API_KEY')
    if not groq_api_key:
        raise ValueError('GROQ_API_KEY is not set in the environment or .env file')

    prompt = f'''Answer the question using only the provided PDF context.
If the context is insufficient, say you do not know.

PDF context:
{context}

Question: {question}'''
    request = Request(
        'https://api.groq.com/openai/v1/chat/completions',
        data=json.dumps({
            'model': GROQ_MODEL,
            'messages': [{'role': 'user', 'content': prompt}],
            'max_tokens': 1000,
        }).encode('utf-8'),
        headers={
            'Authorization': f'Bearer {groq_api_key}',
            'Content-Type': 'application/json',
            'User-Agent': 'genai-pdf-rag/1.0',
        },
        method='POST',
    )
    try:
        with urlopen(request) as response:
            result = json.load(response)
    except HTTPError as error:
        details = error.read().decode('utf-8', errors='replace')
        raise RuntimeError(
            f'Groq request failed with HTTP {error.code}: {details}'
        ) from error
    return result['choices'][0]['message']['content']

In [ ]:
question = 'Summarize the main points of this PDF.'
retrieved_documents = vector_store.similarity_search(question, k=5)
context = '\n\n'.join(document.page_content for document in retrieved_documents)

print(f'Question: {question}\n')
print(f'Answer: {ask_groq(question, context)}')